Small notebook for checking schemes and QE XML output consistency with the test-suite cases.  
For running:  
* adapt commands in process file routine
* in `q-e/test-suite/ENVIRONMENT`:
  - uncomment line 13 and comment line 14 
  - make sure that ESPRESSO_BUILD points to a directory with executables
* install xmlschema package in your python environment. 

In [72]:
from xmlschema import XMLSchema, XMLSchemaChildrenValidationError
from xml.etree import ElementTree as et
from pathlib import Path
from glob import glob
import subprocess
import pickle

In [84]:
qeschemas_topdir = Path('../../')
schema=XMLSchema( qeschemas_topdir / 'PW_CPV/test_schemas/qes_dev_250521.xsd', validation='skip')
suite_dir = Path('/home/pietro/repositories/q-e/test-suite')


In [ ]:


commands_fibion=f"""
module load intel/2022.2.1
module load mkl 
module load openmpi4
export SUITE_DIR=/scratch/pdelugas/qe_gitlab/test-suite/
""" 


commands_pietro=f"""
source /home/pietro/intel/oneapi/setvars.sh
export SUITE_DIR=/home/pietro/repositories/q-e/test-suite/
"""

def process_file(file, arg1, directory, schema, outdir,test_suite_dir='/scratch/pdelugas/qe_gitlab/test-suite',
				 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, commands=commands_fibion):
	input_file = str(directory / file)
	print(input_file)
	commands += f"""
  cd $SUITE_DIR
  source ENVIRONMENT
  export QE_USE_MPI=4
  $SUITE_DIR/run-pw.sh {arg1} {input_file} o e
	"""
	subprocess.run(['bash', '-c', commands], stdout=stdout, stderr=stderr)
	errors = schema.decode(outdir / 'pwscf.xml', validation='lax')[1:]
	if len(errors[0]) > 0:
		print(errors[0][0].msg)
		return (file, errors)
	else:
		print (f"{file} run is xml ok")
	return None



In [86]:

def input_args(subdir):
	jobconfig={'pw_berry':[('berry.in', '1'), ('berry-1.in', '1'), ('berry-2.in','1')],
						 'pw_noncolin':[('noncolin.in' ,'1'), ('noncolin-1.in' ,'1'), ('noncolin-2.in' ,'1'), ('noncolin-cg.in' ,'1'), ('noncolin-rmm.in','1'), 
											('noncolin-constrain_angle.in' ,'1'), ('noncolin-constrain_atomic.in' ,'1'), ('noncolin-constrain_total.in' ,'1'), 
											('noncolin-hyb.in' ,'1'), ('noncolin-pbe.in','1')  ],
						 'pw_metal': [ ('metal.in' ,'1'), ('metal-2.in' ,'1'), ('metal-fermi_dirac.in' ,'1'), ('metal-gaussian.in' ,'1'), ('metal-tetrahedra.in' ,'1'),
										 ('metal-tetrahedra-1.in' ,'1'), ('metal-tetrahedra-2.in','1') ],
						 'pw_electric': [('electric.in' ,'1'), ('electric-1.in' ,'1'), ('electric-2.in' ,'1')],
						 'pw_twochem': [('scf_twochem.in', '1'), ('nscf_twochem.in','2'), ('relax_twochem.in','1'), ('vc-relax_twochem.in','1') ],
						 'pw_spinorbit':[ ('spinorbit.in' ,'1'), ('spinorbit-1.in' ,'1'), ('spinorbit-3.in' ,'1'), ('spinorbit-pbe.in' ,'1'), 
											 ('spinorbit-paw.in','1') ],
						 'pw_uspp': [ ('uspp.in' ,'1'), ('uspp-2.in' ,'1'), ('uspp-cg.in' ,'1'), ('uspp-cg-gamma.in' ,'1'), ('uspp-hyb-g.in' ,'1'),
									  ('uspp-hyb-k.in','1'), ('uspp-mixing_TF.in' ,'1'), ('uspp-mixing_localTF.in' ,'1'), ('uspp-mixing_ndim.in' ,'1'), 
										('uspp-singlegrid.in' ,'1'), ('uspp1-coulomb.in' ,'1'), ('uspp1.in' ,'1'), ('uspp2.in' ,'1'), ('uspp-paro-gamma.in','1'), 
										('uspp-paro-k.in','1'), ('uspp-ppcg-gamma.in','1'), ('uspp-ppcg-k.in','1')  ],
						 'pw_electric' : [ ('electric.in' ,'1'), ('electric-1.in' ,'1'), ('electric-2.in' ,'1') ],
						 'pw_scf'      : [ ('scf.in' ,'1'), ('scf-1.in' ,'1'), ('scf-2.in' ,'1'), ('scf-allfrac.in' ,'1'), ('scf-cg.in' ,'1'), 
						                   ('scf-cg-gamma.in' ,'1'), ('scf-disk_io.in' ,'1'), ('scf-disk_io-1.in' ,'1'), ('scf-disk_io-2.in' ,'1'), ('scf-gamma.in' ,'1'),
										   ('scf-k0.in' ,'1'), ('scf-kauto.in' ,'1'), ('scf-kcrys.in' ,'1'), ('scf-mixing_TF.in' ,'1'), ('scf-mixing_beta.in' ,'1'), 
										   ('scf-mixing_localTF.in' ,'1'), ('scf-mixing_ndim.in' ,'1'), ('scf-ncpp.in' ,'1'), ('scf-nofrac.in' ,'1'), ('scf-occ.in' ,'1'), 
										   ('scf-paro-gamma.in','1'), ('scf-paro-k.in','1'), ('scf-rmm-gamma.in','1'), ('scf-rmm-paro-gamma.in','1'), ('scf-rmm-k.in','1'), 
										   ('scf-rmm-paro-k.in','1'), ('scf-ppcg-gamma.in','1'), ('scf-ppcg-k.in','1'), ('scf-gth.in','1')  ]
						}
	print (subdir.name)
	if subdir.name in jobconfig.keys():
		return jobconfig[subdir.name]
	else:
		return [(file,1) for file in subdir.glob('*.in') if (not file.name.startswith('benchmark')) and 
					                                                       (not file.name.startswith('test'))] 

In [ ]:
from pathlib import Path
directory = suite_dir
outdir = directory
results = [
	process_file(file, arg1, subdir, schema, outdir, commands=commands_pietro)
	for subdir in [_ for _ in directory.glob('pw_*')] 
	for file, arg1 in  input_args(subdir) 
]

issues = [result for result in results if result is not None]

pw_hse
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si444.in
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si444.in run is xml ok
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si111.in
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si111.in run is xml ok
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si222.in
/home/pietro/repositories/q-e/test-suite/pw_hse/hse-si222.in run is xml ok
pw_twochem
/home/pietro/repositories/q-e/test-suite/pw_twochem/scf_twochem.in
scf_twochem.in run is xml ok
/home/pietro/repositories/q-e/test-suite/pw_twochem/nscf_twochem.in
nscf_twochem.in run is xml ok
/home/pietro/repositories/q-e/test-suite/pw_twochem/relax_twochem.in
relax_twochem.in run is xml ok
/home/pietro/repositories/q-e/test-suite/pw_twochem/vc-relax_twochem.in
vc-relax_twochem.in run is xml ok
pw_lda+U
/home/pietro/repositories/q-e/test-suite/pw_lda+U/lda+U_orbital_resolved.in
/home/pietro/repositories/q-e/test-suite/pw_lda+U/lda+U_orbital_resolved.in run is xml ok

In [77]:
from datetime import datetime 
date = datetime.today()
y,d,h,m,s = date.year, date.day, date.hour, date.minute, date.second
with open(f'issues_{y}_{d}_{h}_{m}_{s}.pkl', 'wb') as f:
	pickle.dump([_[0] for _ in issues], f)

In [78]:

def filter_cases(cases, message):
    return [case for case in cases
            if True in [message in _
                        for _ in [__.msg for __ in case[1][0]]
                        ]
    ]
def extract_issues(cases):
    from itertools import chain
    return set([msg  for msg in [msg_ for msg_ in [__[1][0][0].msg for __ in cases]]])
               

In [79]:
with open(f'checkfile.txt', 'w') as checkfile:
	process_file('scf-2.in', '1', directory / 'pw_scf', schema, outdir, stdout=checkfile, stderr=checkfile, commands=commands_pietro)

/home/pietro/repositories/q-e/test-suite/pw_scf/scf-2.in
scf-2.in run is xml ok


In [80]:
cases = extract_issues(issues)

In [83]:
len(cases)
print(list(cases)[0])

failed validating <Element 'dftU' at 0x7a92c4ea0a40> with XsdGroup(model='sequence', occurs=[1, 1]):

Reason: Unexpected child with tag 'Hubbard_Um' at position 2.

Schema component:

  <complexType xmlns="http://www.w3.org/2001/XMLSchema" name="dftUType">
    <sequence>
      <element type="unsignedByte" name="lda_plus_u_kind" minOccurs="0" />
      <element type="qes:HubbardOccType" name="Hubbard_Occ" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:HubbardCommonType" name="Hubbard_U" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:HubbardCommonType" name="Hubbard_J0" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:HubbardCommonType" name="Hubbard_alpha" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:HubbardCommonType" name="Hubbard_beta" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:HubbardJType" name="Hubbard_J" minOccurs="0" maxOccurs="unbounded" />
      <element type="qes:starting_nsType" name="sta